In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import joblib

from surprise import Dataset, Reader, SVD
from surprise.model_selection import train_test_split
from surprise import accuracy

BASE_DIR = Path.cwd().parent

RATINGS_PATH = BASE_DIR / "data" / "raw" / "ratings.csv"

ratings = pd.read_csv(RATINGS_PATH)

print("Ratings:", ratings.shape)
display(ratings.head())

Ratings: (100836, 4)


,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [2]:
reader = Reader(
    rating_scale=(0.5, 5.0)
)

data = Dataset.load_from_df(
    ratings[["userId", "movieId", "rating"]],
    reader
)

trainset, testset = train_test_split(
    data,
    test_size=0.2,
    random_state=42
)

print("Training ratings:", trainset.n_ratings)
print("Test ratings:", len(testset))

Training ratings: 80668
Test ratings: 20168


In [3]:
svd_model = SVD(
    n_factors=100,
    n_epochs=20,
    lr_all=0.005,
    reg_all=0.02,
    random_state=42
)

svd_model.fit(trainset)

print("✅ SVD trained")

✅ SVD trained


In [4]:
predictions = svd_model.test(testset)

rmse = accuracy.rmse(
    predictions,
    verbose=False
)

mae = accuracy.mae(
    predictions,
    verbose=False
)

print(f"RMSE: {rmse:.4f}")
print(f"MAE:  {mae:.4f}")

RMSE: 0.8807
MAE:  0.6766


In [5]:
MODEL_DIR = BASE_DIR / "models"
MODEL_DIR.mkdir(exist_ok=True)

joblib.dump(
    svd_model,
    MODEL_DIR / "svd_model.joblib"
)

print("✅ SVD model saved")

✅ SVD model saved


In [6]:
Get-ChildItem models

SyntaxError: invalid syntax (279386941.py, line 1)

In [7]:
weights = [
    (0.9, 0.1),
    (0.8, 0.2),
    (0.7, 0.3),
    (0.6, 0.4),
    (0.5, 0.5),
    (0.4, 0.6),
    (0.3, 0.7),
    (0.2, 0.8),
    (0.1, 0.9),
]

results = []

for content_weight, collaborative_weight in weights:
    results.append({
        "content_weight": content_weight,
        "collaborative_weight": collaborative_weight,
        "weight_sum": content_weight + collaborative_weight
    })

weights_df = pd.DataFrame(results)

display(weights_df)

,content_weight,collaborative_weight,weight_sum
0,0.9,0.1,1.0
1,0.8,0.2,1.0
2,0.7,0.3,1.0
3,0.6,0.4,1.0
4,0.5,0.5,1.0
5,0.4,0.6,1.0
6,0.3,0.7,1.0
7,0.2,0.8,1.0
8,0.1,0.9,1.0


In [8]:
K = 10
RELEVANCE_THRESHOLD = 4.0

print(f"K = {K}")
print(f"Relevant rating >= {RELEVANCE_THRESHOLD}")

K = 10
Relevant rating >= 4.0


In [9]:
def precision_recall_f1_at_k(
    recommended_ids,
    relevant_ids,
    k=10
):
    recommended_ids = list(recommended_ids)[:k]
    relevant_ids = set(relevant_ids)

    if not recommended_ids:
        return 0.0, 0.0, 0.0

    hits = len(
        set(recommended_ids) & relevant_ids
    )

    precision = hits / len(recommended_ids)

    recall = (
        hits / len(relevant_ids)
        if relevant_ids
        else 0.0
    )

    if precision + recall == 0:
        f1 = 0.0
    else:
        f1 = (
            2 * precision * recall
            / (precision + recall)
        )

    return precision, recall, f1

In [10]:
test_recommended = [1, 2, 3, 4, 5]
test_relevant = [2, 4, 8, 10]

precision, recall, f1 = precision_recall_f1_at_k(
    test_recommended,
    test_relevant,
    k=5
)

print("Precision:", precision)
print("Recall:", recall)
print("F1:", f1)

Precision: 0.4
Recall: 0.5
F1: 0.4444444444444445


In [11]:
# Build a lookup of relevant movies from the test set
test_df = pd.DataFrame(
    testset,
    columns=["userId", "movieId", "actual_rating"]
)

test_df["relevant"] = (
    test_df["actual_rating"] >= RELEVANCE_THRESHOLD
)

relevant_by_user = (
    test_df[test_df["relevant"]]
    .groupby("userId")["movieId"]
    .apply(set)
    .to_dict()
)

print("Test users:", len(relevant_by_user))
print("Relevant interactions:", test_df["relevant"].sum())

Test users: 595
Relevant interactions: 9712


In [12]:
def hybrid_scores_for_user(
    user_id,
    content_weight,
    collaborative_weight
):
    # SVD scores
    svd_scores = []

    for movie_id in df["id"]:
        prediction = svd_model.predict(
            user_id,
            int(movie_id)
        )

        svd_scores.append(prediction.est)

    scores = pd.DataFrame({
        "movieId": df["id"].values,
        "content_score": np.zeros(len(df)),
        "collaborative_score": svd_scores
    })

    # Normalize collaborative scores
    scores["collaborative_score"] = min_max_normalize(
        scores["collaborative_score"]
    )

    return scores

In [13]:
def evaluate_svd_at_k(
    model,
    test_df,
    k=10,
    threshold=4.0
):
    precisions = []
    recalls = []
    f1_scores = []

    relevant_by_user = (
        test_df[test_df["actual_rating"] >= threshold]
        .groupby("userId")["movieId"]
        .apply(set)
        .to_dict()
    )

    for user_id, relevant_movies in relevant_by_user.items():

        candidate_movies = df["id"].tolist()

        predictions = [
            (
                movie_id,
                model.predict(
                    int(user_id),
                    int(movie_id)
                ).est
            )
            for movie_id in candidate_movies
        ]

        predictions.sort(
            key=lambda x: x[1],
            reverse=True
        )

        recommended = [
            movie_id
            for movie_id, _ in predictions[:k]
        ]

        precision, recall, f1 = (
            precision_recall_f1_at_k(
                recommended,
                relevant_movies,
                k
            )
        )

        precisions.append(precision)
        recalls.append(recall)
        f1_scores.append(f1)

    return {
        "precision@10": np.mean(precisions),
        "recall@10": np.mean(recalls),
        "f1@10": np.mean(f1_scores)
    }

In [14]:
svd_metrics = evaluate_svd_at_k(
    svd_model,
    test_df,
    k=10,
    threshold=4.0
)

print(svd_metrics)

NameError: name 'df' is not defined

In [15]:
import pandas as pd
import numpy as np
import joblib

from pathlib import Path
from surprise import Dataset, Reader, SVD, accuracy
from surprise.model_selection import train_test_split

BASE_DIR = Path.cwd().parent

# Load TMDB features
MOVIES_PATH = (
    BASE_DIR
    / "data"
    / "processed"
    / "movies_features.csv"
)

df = pd.read_csv(MOVIES_PATH)

print("TMDB dataset:", df.shape)
display(df[["id", "title"]].head())

TMDB dataset: (4803, 11)


,id,title
0,19995,Avatar
1,285,Pirates of the Caribbean: At World's End
2,206647,Spectre
3,49026,The Dark Knight Rises
4,49529,John Carter


In [16]:
RATINGS_PATH = BASE_DIR / "data" / "raw" / "ratings.csv"

ratings = pd.read_csv(RATINGS_PATH)

print("Ratings:", ratings.shape)
display(ratings.head())

Ratings: (100836, 4)


,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [17]:
reader = Reader(
    rating_scale=(0.5, 5.0)
)

data = Dataset.load_from_df(
    ratings[["userId", "movieId", "rating"]],
    reader
)

trainset, testset = train_test_split(
    data,
    test_size=0.2,
    random_state=42
)

print("Training ratings:", trainset.n_ratings)
print("Test ratings:", len(testset))

Training ratings: 80668
Test ratings: 20168


In [18]:
svd_model = SVD(
    n_factors=100,
    n_epochs=20,
    lr_all=0.005,
    reg_all=0.02,
    random_state=42
)

svd_model.fit(trainset)

print("✅ SVD model ready")

✅ SVD model ready


In [19]:
predictions = svd_model.test(testset)

rmse = accuracy.rmse(
    predictions,
    verbose=False
)

mae = accuracy.mae(
    predictions,
    verbose=False
)

print(f"RMSE: {rmse:.4f}")
print(f"MAE:  {mae:.4f}")

RMSE: 0.8807
MAE:  0.6766


In [20]:
test_df = pd.DataFrame(
    testset,
    columns=["userId", "movieId", "actual_rating"]
)

print("Test dataframe:", test_df.shape)

display(test_df.head())

Test dataframe: (20168, 3)


,userId,movieId,actual_rating
0,140,6765,3.5
1,603,290,4.0
2,438,5055,4.0
3,433,164179,5.0
4,474,5114,4.0


In [21]:
K = 10
RELEVANCE_THRESHOLD = 4.0

In [22]:
evaluate_svd_at_k(
    svd_model,
    test_df,
    k=10,
    threshold=4.0
)

{'precision@10': np.float64(0.02453781512605042),
 'recall@10': np.float64(0.0254095509671597),
 'f1@10': np.float64(0.020357676182624025)}

In [23]:
df["id"]

0        19995
1          285
2       206647
3        49026
4        49529
         ...  
4798      9367
4799     72766
4800    231617
4801    126186
4802     25975
Name: id, Length: 4803, dtype: int64

In [24]:
LINKS_PATH = BASE_DIR / "data" / "raw" / "links.csv"

links = pd.read_csv(LINKS_PATH)

ratings_tmdb = ratings.merge(
    links[["movieId", "tmdbId"]],
    on="movieId",
    how="left"
)

ratings_tmdb = ratings_tmdb.dropna(
    subset=["tmdbId"]
).copy()

ratings_tmdb["tmdbId"] = (
    ratings_tmdb["tmdbId"]
    .astype(int)
)

In [25]:
ratings_tmdb = ratings_tmdb[
    ratings_tmdb["tmdbId"].isin(df["id"])
].copy()

print(
    "Ratings matching TMDB catalog:",
    len(ratings_tmdb)
)

Ratings matching TMDB catalog: 70194


In [26]:
svd_ratings = ratings_tmdb[
    ["userId", "tmdbId", "rating"]
].rename(
    columns={"tmdbId": "movieId"}
)

In [27]:
reader = Reader(
    rating_scale=(0.5, 5.0)
)

data = Dataset.load_from_df(
    svd_ratings[
        ["userId", "movieId", "rating"]
    ],
    reader
)

trainset, testset = train_test_split(
    data,
    test_size=0.2,
    random_state=42
)

In [28]:
svd_model = SVD(
    n_factors=100,
    n_epochs=20,
    lr_all=0.005,
    reg_all=0.02,
    random_state=42
)

svd_model.fit(trainset)

print("✅ SVD trained on mapped TMDB IDs")

✅ SVD trained on mapped TMDB IDs


In [29]:
test_df = pd.DataFrame(
    testset,
    columns=["userId", "movieId", "actual_rating"]
)

In [30]:
svd_metrics = evaluate_svd_at_k(
    svd_model,
    test_df,
    k=10,
    threshold=4.0
)

print(svd_metrics)

{'precision@10': np.float64(0.02956081081081081), 'recall@10': np.float64(0.030200801496662), 'f1@10': np.float64(0.02429307771646813)}


In [1]:
def evaluate_svd_ranking(
    model,
    test_df,
    catalog_ids,
    k=10,
    threshold=4.0
):
    precisions = []
    recalls = []
    f1_scores = []

    relevant_by_user = (
        test_df[test_df["actual_rating"] >= threshold]
        .groupby("userId")["movieId"]
        .apply(set)
        .to_dict()
    )

    catalog_ids = [int(x) for x in catalog_ids]

    for user_id, relevant_movies in relevant_by_user.items():

        predictions = []

        for movie_id in catalog_ids:
            prediction = model.predict(
                int(user_id),
                int(movie_id)
            )

            predictions.append(
                (movie_id, prediction.est)
            )

        predictions.sort(
            key=lambda x: x[1],
            reverse=True
        )

        recommended = [
            movie_id
            for movie_id, _ in predictions[:k]
        ]

        precision, recall, f1 = (
            precision_recall_f1_at_k(
                recommended,
                relevant_movies,
                k=k
            )
        )

        precisions.append(precision)
        recalls.append(recall)
        f1_scores.append(f1)

    return {
        "precision@10": np.mean(precisions),
        "recall@10": np.mean(recalls),
        "f1@10": np.mean(f1_scores)
    }

In [2]:
from sklearn.preprocessing import normalize

In [3]:
def build_user_content_scores(
    user_id,
    train_ratings,
    df,
    tfidf_matrix
):
    user_ratings = train_ratings[
        train_ratings["userId"] == user_id
    ]

    # Use positively rated movies
    liked_movies = user_ratings[
        user_ratings["rating"] >= 4.0
    ]

    if liked_movies.empty:
        return np.zeros(len(df))

    movie_to_index = {
        int(movie_id): idx
        for idx, movie_id in enumerate(df["id"])
    }

    profile_vectors = []
    weights = []

    for _, row in liked_movies.iterrows():

        movie_id = int(row["movieId"])

        if movie_id not in movie_to_index:
            continue

        idx = movie_to_index[movie_id]

        profile_vectors.append(
            tfidf_matrix[idx]
        )

        weights.append(
            float(row["rating"])
        )

    if not profile_vectors:
        return np.zeros(len(df))

    profile = np.average(
        profile_vectors,
        axis=0,
        weights=weights
    )

    profile = np.asarray(profile).reshape(1, -1)

    profile = normalize(profile)

    content_scores = (
        tfidf_matrix @ profile.T
    ).ravel()

    return content_scores

In [4]:
def get_hybrid_scores(
    user_id,
    train_ratings,
    df,
    tfidf_matrix,
    svd_model,
    content_weight=0.6,
    collaborative_weight=0.4
):

    # Content score
    content_scores = build_user_content_scores(
        user_id,
        train_ratings,
        df,
        tfidf_matrix
    )

    # Collaborative score
    collaborative_scores = np.array([
        svd_model.predict(
            int(user_id),
            int(movie_id)
        ).est
        for movie_id in df["id"]
    ])

    # Normalize both
    content_scores = min_max_normalize(
        pd.Series(content_scores)
    ).to_numpy()

    collaborative_scores = min_max_normalize(
        pd.Series(collaborative_scores)
    ).to_numpy()

    # Hybrid
    hybrid_scores = (
        content_weight * content_scores
        +
        collaborative_weight * collaborative_scores
    )

    return pd.DataFrame({
        "movieId": df["id"].astype(int),
        "content_score": content_scores,
        "collaborative_score": collaborative_scores,
        "hybrid_score": hybrid_scores
    })

In [5]:
train_df = pd.DataFrame(
    trainset.all_ratings(),
    columns=["inner_user", "inner_movie", "rating"]
)

NameError: name 'pd' is not defined

In [6]:
import pandas as pd
import numpy as np
import joblib

from pathlib import Path
from sklearn.preprocessing import normalize

print("✅ Libraries loaded")

✅ Libraries loaded


In [7]:
train_df = pd.DataFrame(
    trainset.all_ratings(),
    columns=["inner_user", "inner_movie", "rating"]
)

display(train_df.head())

NameError: name 'trainset' is not defined

In [8]:
train_df = pd.DataFrame(
    trainset.all_ratings(),
    columns=["inner_user", "inner_movie", "rating"]
)

NameError: name 'trainset' is not defined

In [9]:
print(ratings_tmdb.shape)
print(ratings_tmdb.columns.tolist())
display(ratings_tmdb.head())

NameError: name 'ratings_tmdb' is not defined

In [10]:
import pandas as pd
import numpy as np
import joblib

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import normalize

from surprise import Dataset, Reader, SVD, accuracy

# ============================================================
# PROJECT PATH
# ============================================================

BASE_DIR = Path.cwd().parent

print("Project:", BASE_DIR)

# ============================================================
# LOAD TMDB DATA
# ============================================================

MOVIES_PATH = (
    BASE_DIR
    / "data"
    / "processed"
    / "movies_features.csv"
)

df = pd.read_csv(MOVIES_PATH)

print("TMDB dataset:", df.shape)
display(df[["id", "title"]].head())

# ============================================================
# LOAD MOVIELENS RATINGS
# ============================================================

RATINGS_PATH = (
    BASE_DIR
    / "data"
    / "raw"
    / "ratings.csv"
)

ratings = pd.read_csv(RATINGS_PATH)

print("MovieLens ratings:", ratings.shape)
display(ratings.head())

# ============================================================
# LOAD MOVIELENS → TMDB MAPPING
# ============================================================

LINKS_PATH = (
    BASE_DIR
    / "data"
    / "raw"
    / "links.csv"
)

links = pd.read_csv(LINKS_PATH)

print("Links:", links.shape)
display(links.head())

# ============================================================
# CREATE MAPPING
# ============================================================

ratings_tmdb = ratings.merge(
    links[["movieId", "tmdbId"]],
    on="movieId",
    how="left"
)

ratings_tmdb = ratings_tmdb.dropna(
    subset=["tmdbId"]
).copy()

ratings_tmdb["tmdbId"] = (
    ratings_tmdb["tmdbId"]
    .astype(int)
)

# ============================================================
# KEEP ONLY MOVIES IN OUR TMDB CATALOG
# ============================================================

tmdb_ids = set(
    df["id"].astype(int)
)

ratings_tmdb = ratings_tmdb[
    ratings_tmdb["tmdbId"].isin(tmdb_ids)
].copy()

print(
    "Ratings matching TMDB catalog:",
    len(ratings_tmdb)
)

print(
    "Users:",
    ratings_tmdb["userId"].nunique()
)

print(
    "Movies:",
    ratings_tmdb["tmdbId"].nunique()
)

display(ratings_tmdb.head())

Project: c:\Users\Win 10\Desktop\movie-recommender
TMDB dataset: (4803, 11)


,id,title
0,19995,Avatar
1,285,Pirates of the Caribbean: At World's End
2,206647,Spectre
3,49026,The Dark Knight Rises
4,49529,John Carter


MovieLens ratings: (100836, 4)


,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


Links: (9742, 3)


,movieId,imdbId,tmdbId
0,1,114709,862.0
1,2,113497,8844.0
2,3,113228,15602.0
3,4,114885,31357.0
4,5,113041,11862.0


Ratings matching TMDB catalog: 70194
Users: 610
Movies: 3535


,userId,movieId,rating,timestamp,tmdbId
0,1,1,4.0,964982703,862
3,1,47,5.0,964983815,807
4,1,50,5.0,964982931,629
5,1,70,3.0,964982400,755
6,1,101,5.0,964980868,13685


In [11]:
train_ratings, test_ratings = train_test_split(
    ratings_tmdb,
    test_size=0.2,
    random_state=42
)

print("Training ratings:", len(train_ratings))
print("Test ratings:", len(test_ratings))

print(
    "Training users:",
    train_ratings["userId"].nunique()
)

print(
    "Training movies:",
    train_ratings["tmdbId"].nunique()
)

display(train_ratings.head())

Training ratings: 56155
Test ratings: 14039
Training users: 610
Training movies: 3413


,userId,movieId,rating,timestamp,tmdbId
66510,428,2671,1.0,1111489708,509
64596,414,51084,3.0,1199721810,11172
74431,474,5377,5.0,1087831477,245
36505,249,1722,3.5,1440669132,714
533,5,296,5.0,847434748,680


In [12]:
reader = Reader(
    rating_scale=(0.5, 5.0)
)

svd_data = Dataset.load_from_df(
    train_ratings[
        ["userId", "tmdbId", "rating"]
    ].rename(
        columns={"tmdbId": "movieId"}
    ),
    reader
)

trainset = svd_data.build_full_trainset()

svd_model = SVD(
    n_factors=100,
    n_epochs=20,
    lr_all=0.005,
    reg_all=0.02,
    random_state=42
)

svd_model.fit(trainset)

print("✅ SVD trained on training data only")

✅ SVD trained on training data only


In [13]:
test_df = test_ratings[
    ["userId", "tmdbId", "rating"]
].copy()

test_df = test_df.rename(
    columns={"rating": "actual_rating"}
)

print("Evaluation dataset:", test_df.shape)

display(test_df.head())

Evaluation dataset: (14039, 3)


,userId,tmdbId,actual_rating
18839,122,1572,5.0
45773,304,9271,4.0
99495,608,1858,5.0
84347,541,1909,4.0
30710,215,36557,4.0


In [14]:
print("df:", df.shape)
print("ratings_tmdb:", ratings_tmdb.shape)
print("train_ratings:", train_ratings.shape)
print("test_ratings:", test_ratings.shape)
print("test_df:", test_df.shape)
print("SVD:", type(svd_model).__name__)

df: (4803, 11)
ratings_tmdb: (70194, 5)
train_ratings: (56155, 5)
test_ratings: (14039, 5)
test_df: (14039, 3)
SVD: SVD


In [15]:
print(type(tfidf_matrix))
print(tfidf_matrix.shape)

NameError: name 'tfidf_matrix' is not defined

In [16]:
print(df.columns.tolist())

['id', 'title', 'overview', 'genres', 'keywords', 'cast', 'crew', 'vote_average', 'vote_count', 'popularity', 'tags']


In [17]:
print(df["tags"].head())

0    in the 22nd century a paraplegic marine is dis...
1    captain barbossa long believed to be dead has ...
2    a cryptic message from bond s past sends him o...
3    following the death of district attorney harve...
4    john carter is a war weary former military cap...
Name: tags, dtype: str


In [18]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=5000,
    stop_words="english"
)

tfidf_matrix = tfidf.fit_transform(
    df["tags"].fillna("")
)

print("TF-IDF matrix:", tfidf_matrix.shape)

TF-IDF matrix: (4803, 5000)


In [19]:
print(type(tfidf_matrix))
print(tfidf_matrix.shape)

<class 'scipy.sparse._csr.csr_matrix'>
(4803, 5000)


In [20]:
user_1_content = build_user_content_scores(
    user_id=1,
    train_ratings=train_ratings,
    df=df,
    tfidf_matrix=tfidf_matrix
)

print("Number of scores:", len(user_1_content))
print("Best score:", user_1_content.max())

ValueError: setting an array element with a sequence.

In [21]:
def build_user_content_scores(
    user_id,
    train_ratings,
    df,
    tfidf_matrix
):
    # Ratings for this user from TRAIN only
    user_ratings = train_ratings[
        train_ratings["userId"] == user_id
    ]

    # Movies the user liked
    liked_movies = user_ratings[
        user_ratings["rating"] >= 4.0
    ]

    if liked_movies.empty:
        return np.zeros(len(df))

    # TMDB ID -> TF-IDF row index
    movie_to_index = {
        int(movie_id): idx
        for idx, movie_id in enumerate(df["id"])
    }

    profile_rows = []
    weights = []

    for _, row in liked_movies.iterrows():

        tmdb_id = int(row["tmdbId"])

        if tmdb_id not in movie_to_index:
            continue

        idx = movie_to_index[tmdb_id]

        profile_rows.append(
            tfidf_matrix[idx]
        )

        weights.append(
            float(row["rating"])
        )

    if not profile_rows:
        return np.zeros(len(df))

    # Stack sparse TF-IDF rows
    profile_matrix = np.vstack([
        row.toarray().ravel()
        for row in profile_rows
    ])

    # Weighted user profile
    profile = np.average(
        profile_matrix,
        axis=0,
        weights=weights
    )

    # Normalize profile
    profile_norm = np.linalg.norm(profile)

    if profile_norm == 0:
        return np.zeros(len(df))

    profile = profile / profile_norm

    # Calculate cosine similarity
    content_scores = (
        tfidf_matrix @ profile
    )

    return np.asarray(content_scores).ravel()

In [22]:
user_1_content = build_user_content_scores(
    user_id=1,
    train_ratings=train_ratings,
    df=df,
    tfidf_matrix=tfidf_matrix
)

print("Number of scores:", len(user_1_content))
print("Best score:", user_1_content.max())

Number of scores: 4803
Best score: 0.294088698597289


In [23]:
top_indices = np.argsort(
    user_1_content
)[::-1][:10]

user_1_content_recommendations = df.iloc[
    top_indices
][["id", "title"]].copy()

user_1_content_recommendations["content_score"] = (
    user_1_content[top_indices]
)

display(user_1_content_recommendations)

,id,title,content_score
3143,667,You Only Live Twice,0.294089
3343,253,Live and Let Die,0.286726
4339,646,Dr. No,0.273710
4071,657,From Russia with Love,0.265997
1490,1892,Return of the Jedi,0.259096
2912,11,Star Wars,0.255548
4401,43630,The Helix... Loaded,0.240504
3884,658,Goldfinger,0.235656
3251,668,On Her Majesty's Secret Service,0.228313
1006,89,Indiana Jones and the Last Crusade,0.226838


In [24]:
def get_hybrid_scores(
    user_id,
    train_ratings,
    df,
    tfidf_matrix,
    svd_model,
    content_weight=0.6,
    collaborative_weight=0.4
):
    # -----------------------------
    # Content-based scores
    # -----------------------------
    content_scores = build_user_content_scores(
        user_id=user_id,
        train_ratings=train_ratings,
        df=df,
        tfidf_matrix=tfidf_matrix
    )

    # -----------------------------
    # Collaborative scores
    # -----------------------------
    collaborative_scores = np.array([
        svd_model.predict(
            int(user_id),
            int(movie_id)
        ).est
        for movie_id in df["id"]
    ])

    # -----------------------------
    # Normalize scores
    # -----------------------------
    content_series = pd.Series(
        content_scores
    )

    collaborative_series = pd.Series(
        collaborative_scores
    )

    content_normalized = (
        min_max_normalize(content_series)
        .to_numpy()
    )

    collaborative_normalized = (
        min_max_normalize(collaborative_series)
        .to_numpy()
    )

    # -----------------------------
    # Hybrid score
    # -----------------------------
    hybrid_scores = (
        content_weight * content_normalized
        +
        collaborative_weight * collaborative_normalized
    )

    return pd.DataFrame({
        "movieId": df["id"].astype(int),
        "content_score": content_normalized,
        "collaborative_score": collaborative_normalized,
        "hybrid_score": hybrid_scores
    })

In [25]:
user_1_hybrid = get_hybrid_scores(
    user_id=1,
    train_ratings=train_ratings,
    df=df,
    tfidf_matrix=tfidf_matrix,
    svd_model=svd_model,
    content_weight=0.6,
    collaborative_weight=0.4
)

NameError: name 'min_max_normalize' is not defined

In [26]:
def min_max_normalize(series):
    min_value = series.min()
    max_value = series.max()

    if max_value == min_value:
        return pd.Series(
            0.0,
            index=series.index
        )

    return (
        (series - min_value)
        / (max_value - min_value)
    )

print("✅ Normalization function ready")

✅ Normalization function ready


In [27]:
def get_hybrid_scores(...):
    ...

SyntaxError: invalid syntax (3421164053.py, line 1)

In [28]:
def min_max_normalize(series):
    min_value = series.min()
    max_value = series.max()

    if max_value == min_value:
        return pd.Series(0.0, index=series.index)

    return (series - min_value) / (max_value - min_value)

print("✅ Normalization function ready")

✅ Normalization function ready


In [29]:
def get_hybrid_scores(
    user_id,
    train_ratings,
    df,
    tfidf_matrix,
    svd_model,
    content_weight=0.6,
    collaborative_weight=0.4
):
    ...

In [30]:
user_1_hybrid = get_hybrid_scores(
    user_id=1,
    train_ratings=train_ratings,
    df=df,
    tfidf_matrix=tfidf_matrix,
    svd_model=svd_model,
    content_weight=0.6,
    collaborative_weight=0.4
)

print("✅ Hybrid scores generated")
print(user_1_hybrid.shape)

✅ Hybrid scores generated


AttributeError: 'NoneType' object has no attribute 'shape'

In [31]:
def get_hybrid_scores(
    user_id,
    train_ratings,
    df,
    tfidf_matrix,
    svd_model,
    content_weight=0.6,
    collaborative_weight=0.4
):
    # Content-based scores
    content_scores = build_user_content_scores(
        user_id=user_id,
        train_ratings=train_ratings,
        df=df,
        tfidf_matrix=tfidf_matrix
    )

    # Collaborative scores
    collaborative_scores = np.array([
        svd_model.predict(
            int(user_id),
            int(movie_id)
        ).est
        for movie_id in df["id"]
    ])

    # Normalize
    content_normalized = min_max_normalize(
        pd.Series(content_scores)
    ).to_numpy()

    collaborative_normalized = min_max_normalize(
        pd.Series(collaborative_scores)
    ).to_numpy()

    # Hybrid score
    hybrid_scores = (
        content_weight * content_normalized
        + collaborative_weight * collaborative_normalized
    )

    # IMPORTANT: return the dataframe
    return pd.DataFrame({
        "movieId": df["id"].astype(int),
        "content_score": content_normalized,
        "collaborative_score": collaborative_normalized,
        "hybrid_score": hybrid_scores
    })

In [32]:
user_1_hybrid = get_hybrid_scores(
    user_id=1,
    train_ratings=train_ratings,
    df=df,
    tfidf_matrix=tfidf_matrix,
    svd_model=svd_model,
    content_weight=0.6,
    collaborative_weight=0.4
)

print("✅ Hybrid scores generated")
print(user_1_hybrid.shape)

✅ Hybrid scores generated
(4803, 4)


In [33]:
top_hybrid = (
    user_1_hybrid
    .sort_values("hybrid_score", ascending=False)
    .head(10)
)

top_hybrid = top_hybrid.merge(
    df[["id", "title", "vote_average", "popularity"]],
    left_on="movieId",
    right_on="id",
    how="left"
)

display(
    top_hybrid[
        [
            "title",
            "hybrid_score",
            "content_score",
            "collaborative_score",
            "vote_average",
            "popularity"
        ]
    ]
)

,title,hybrid_score,content_score,collaborative_score,vote_average,popularity
0,Star Wars,0.902684,0.868949,0.953286,8.1,126.393695
1,Return of the Jedi,0.902227,0.881013,0.934049,7.9,46.509071
2,Live and Let Die,0.899966,0.974964,0.787470,6.4,30.465138
3,Dr. No,0.876579,0.930705,0.795391,6.9,48.901542
4,You Only Live Twice,0.857977,1.000000,0.644943,6.5,28.675891
5,From Russia with Love,0.837427,0.904479,0.736849,6.9,41.298723
6,Indiana Jones and the Last Crusade,0.829625,0.771324,0.917078,7.6,80.972475
7,Goldfinger,0.808445,0.801308,0.819152,7.2,47.812466
8,The Empire Strikes Back,0.793357,0.728720,0.890313,8.2,78.517830
9,The Terminator,0.785084,0.741407,0.850599,7.3,74.234793


In [34]:
def evaluate_hybrid_at_k(
    content_weight,
    collaborative_weight,
    train_ratings,
    test_df,
    df,
    tfidf_matrix,
    svd_model,
    k=10,
    threshold=4.0
):
    precisions = []
    recalls = []
    f1_scores = []

    relevant_by_user = (
        test_df[test_df["actual_rating"] >= threshold]
        .groupby("userId")["tmdbId"]
        .apply(set)
        .to_dict()
    )

    for user_id, relevant_movies in relevant_by_user.items():

        hybrid_df = get_hybrid_scores(
            user_id=user_id,
            train_ratings=train_ratings,
            df=df,
            tfidf_matrix=tfidf_matrix,
            svd_model=svd_model,
            content_weight=content_weight,
            collaborative_weight=collaborative_weight
        )

        recommended = (
            hybrid_df
            .sort_values(
                "hybrid_score",
                ascending=False
            )
            .head(k)["movieId"]
            .tolist()
        )

        precision, recall, f1 = (
            precision_recall_f1_at_k(
                recommended,
                relevant_movies,
                k=k
            )
        )

        precisions.append(precision)
        recalls.append(recall)
        f1_scores.append(f1)

    return {
        "content_weight": content_weight,
        "collaborative_weight": collaborative_weight,
        "precision@10": np.mean(precisions),
        "recall@10": np.mean(recalls),
        "f1@10": np.mean(f1_scores)
    }

In [35]:
weights = [
    (0.9, 0.1),
    (0.8, 0.2),
    (0.7, 0.3),
    (0.6, 0.4),
    (0.5, 0.5),
    (0.4, 0.6),
    (0.3, 0.7),
    (0.2, 0.8),
    (0.1, 0.9),
]

results = []

for content_weight, collaborative_weight in weights:

    print(
        f"Testing {content_weight:.0%} "
        f"Content / {collaborative_weight:.0%} SVD..."
    )

    result = evaluate_hybrid_at_k(
        content_weight=content_weight,
        collaborative_weight=collaborative_weight,
        train_ratings=train_ratings,
        test_df=test_df,
        df=df,
        tfidf_matrix=tfidf_matrix,
        svd_model=svd_model,
        k=10,
        threshold=4.0
    )

    results.append(result)

results_df = pd.DataFrame(results)

display(results_df)

Testing 90% Content / 10% SVD...


NameError: name 'precision_recall_f1_at_k' is not defined

In [36]:
def precision_recall_f1_at_k(
    recommended_ids,
    relevant_ids,
    k=10
):
    recommended_ids = list(recommended_ids)[:k]
    relevant_ids = set(relevant_ids)

    if not recommended_ids:
        return 0.0, 0.0, 0.0

    hits = len(
        set(recommended_ids) & relevant_ids
    )

    precision = hits / len(recommended_ids)

    recall = (
        hits / len(relevant_ids)
        if relevant_ids
        else 0.0
    )

    if precision + recall == 0:
        f1 = 0.0
    else:
        f1 = (
            2 * precision * recall
            / (precision + recall)
        )

    return precision, recall, f1

print("✅ Precision / Recall / F1 function ready")

✅ Precision / Recall / F1 function ready


In [37]:
weights = [
    (0.9, 0.1),
    (0.8, 0.2),
    (0.7, 0.3),
    (0.6, 0.4),
    (0.5, 0.5),
    (0.4, 0.6),
    (0.3, 0.7),
    (0.2, 0.8),
    (0.1, 0.9),
]

results = []

for content_weight, collaborative_weight in weights:

    print(
        f"Testing {content_weight:.0%} "
        f"Content / {collaborative_weight:.0%} SVD..."
    )

    result = evaluate_hybrid_at_k(
        content_weight=content_weight,
        collaborative_weight=collaborative_weight,
        train_ratings=train_ratings,
        test_df=test_df,
        df=df,
        tfidf_matrix=tfidf_matrix,
        svd_model=svd_model,
        k=10,
        threshold=4.0
    )

    results.append(result)

results_df = pd.DataFrame(results)

display(results_df)

Testing 90% Content / 10% SVD...
Testing 80% Content / 20% SVD...
Testing 70% Content / 30% SVD...
Testing 60% Content / 40% SVD...
Testing 50% Content / 50% SVD...
Testing 40% Content / 60% SVD...
Testing 30% Content / 70% SVD...
Testing 20% Content / 80% SVD...
Testing 10% Content / 90% SVD...


,content_weight,collaborative_weight,precision@10,recall@10,f1@10
0,0.9,0.1,0.008968,0.013406,0.007638
1,0.8,0.2,0.008799,0.013979,0.007925
2,0.7,0.3,0.009137,0.014259,0.008331
3,0.6,0.4,0.008968,0.015788,0.008388
4,0.5,0.5,0.009306,0.018472,0.009459
5,0.4,0.6,0.011168,0.021098,0.011304
6,0.3,0.7,0.012860,0.021549,0.012687
7,0.2,0.8,0.016074,0.023894,0.015393
8,0.1,0.9,0.024027,0.029253,0.021134


In [38]:
results_df.sort_values(
    "f1@10",
    ascending=False
)

,content_weight,collaborative_weight,precision@10,recall@10,f1@10
8,0.1,0.9,0.024027,0.029253,0.021134
7,0.2,0.8,0.016074,0.023894,0.015393
6,0.3,0.7,0.012860,0.021549,0.012687
5,0.4,0.6,0.011168,0.021098,0.011304
4,0.5,0.5,0.009306,0.018472,0.009459
3,0.6,0.4,0.008968,0.015788,0.008388
2,0.7,0.3,0.009137,0.014259,0.008331
1,0.8,0.2,0.008799,0.013979,0.007925
0,0.9,0.1,0.008968,0.013406,0.007638


In [39]:
best_model = results_df.loc[
    results_df["f1@10"].idxmax()
]

print("🏆 BEST HYBRID CONFIGURATION")
display(best_model.to_frame().T)

🏆 BEST HYBRID CONFIGURATION


,content_weight,collaborative_weight,precision@10,recall@10,f1@10
8,0.1,0.9,0.024027,0.029253,0.021134


In [40]:
BEST_CONTENT_WEIGHT = 0.1
BEST_COLLABORATIVE_WEIGHT = 0.9

print(
    f"Best configuration: "
    f"{BEST_CONTENT_WEIGHT:.0%} Content / "
    f"{BEST_COLLABORATIVE_WEIGHT:.0%} Collaborative"
)

Best configuration: 10% Content / 90% Collaborative


In [41]:
import json

config = {
    "content_weight": BEST_CONTENT_WEIGHT,
    "collaborative_weight": BEST_COLLABORATIVE_WEIGHT,
    "k": 10,
    "relevance_threshold": 4.0,
    "precision_at_10": 0.024027,
    "recall_at_10": 0.029253,
    "f1_at_10": 0.021134
}

CONFIG_PATH = BASE_DIR / "models" / "hybrid_config.json"

with open(CONFIG_PATH, "w", encoding="utf-8") as f:
    json.dump(config, f, indent=4)

print(f"✅ Configuration saved to: {CONFIG_PATH}")

✅ Configuration saved to: c:\Users\Win 10\Desktop\movie-recommender\models\hybrid_config.json


In [42]:
print(CONFIG_PATH.exists())

True


In [43]:
import joblib

TFIDF_VECTOR_PATH = BASE_DIR / "models" / "tfidf_vectorizer.joblib"

joblib.dump(
    tfidf,
    TFIDF_VECTOR_PATH
)

print(f"✅ TF-IDF vectorizer saved:")
print(TFIDF_VECTOR_PATH)

✅ TF-IDF vectorizer saved:
c:\Users\Win 10\Desktop\movie-recommender\models\tfidf_vectorizer.joblib


In [44]:
from scipy.sparse import save_npz

TFIDF_MATRIX_PATH = BASE_DIR / "models" / "tfidf_matrix.npz"

save_npz(
    TFIDF_MATRIX_PATH,
    tfidf_matrix
)

print(f"✅ TF-IDF matrix saved:")
print(TFIDF_MATRIX_PATH)

✅ TF-IDF matrix saved:
c:\Users\Win 10\Desktop\movie-recommender\models\tfidf_matrix.npz


In [45]:
from pathlib import Path

model_files = [
    BASE_DIR / "models" / "svd_model.joblib",
    BASE_DIR / "models" / "hybrid_config.json",
    BASE_DIR / "models" / "tfidf_vectorizer.joblib",
    BASE_DIR / "models" / "tfidf_matrix.npz",
]

for path in model_files:
    print(
        "✅" if path.exists() else "❌",
        path.name,
        f"({path.stat().st_size:,} bytes)"
        if path.exists()
        else ""
    )

✅ svd_model.joblib (10,020,435 bytes)
✅ hybrid_config.json (204 bytes)
✅ tfidf_vectorizer.joblib (182,940 bytes)
✅ tfidf_matrix.npz (1,478,504 bytes)


In [46]:
TRAIN_RATINGS_PATH = (
    BASE_DIR
    / "data"
    / "processed"
    / "train_ratings.csv"
)

train_ratings.to_csv(
    TRAIN_RATINGS_PATH,
    index=False
)

print(
    f"✅ Training ratings saved:"
    f"\n{TRAIN_RATINGS_PATH}"
)

print(
    "Rows:",
    len(train_ratings)
)

✅ Training ratings saved:
c:\Users\Win 10\Desktop\movie-recommender\data\processed\train_ratings.csv
Rows: 56155
